In [39]:
import random
import pandas as pd

In [65]:
# Path to the dataset
df = pd.read_csv('../dataset/sms/test.csv')

df[:20]

,email,target
0,"Oh right, ok. I'll make sure that i do loads o...",ham
1,I am in tirupur. call you da.,ham
2,No that just means you have a fat head,ham
3,"You have won ?1,000 cash or a ?2,000 prize! To...",spam
4,Come aftr &lt;DECIMAL&gt; ..now i m cleaning t...,ham
5,Friendship poem: Dear O Dear U R Not Near But ...,ham
6,Wot about on wed nite I am 3 then but only til 9!,ham
7,Dont talk to him ever ok its my word.,ham
8,Congrats kano..whr s the treat maga?,ham
9,Eh u remember how 2 spell his name... Yes i di...,ham


In [3]:
df.columns

Index(['email', 'target'], dtype='object')

In [66]:
df2 = pd.read_csv('../dataset/sms/val.csv')

In [67]:
df2.head()

,email,target
0,have got * few things to do. may be in * pub l...,ham
1,Okie but i scared u say i fat... Then u dun wa...,ham
2,Wot is u up 2 then bitch?,ham
3,Ill be at yours in about 3 mins but look out f...,ham
4,Becoz its &lt;#&gt; jan whn al the post ofice ...,ham


In [68]:
df3 = pd.read_csv('../dataset/sms/train.csv')

In [8]:
df['target'].str.lower().eq('ham').sum()

706

In [9]:
df['target'].str.lower().eq('spam').sum()

293

In [10]:
df2['target'].str.lower().eq('ham').sum()

565

In [11]:
df2['target'].str.lower().eq('spam').sum()

234

In [12]:
df3['target'].str.lower().eq('ham').sum()

2260

In [13]:
df3['target'].str.lower().eq('spam').sum()

936

In [14]:
# Define Obfuscation Function
def obfuscate_word(word):
    substitutions = {'a':'@', 'e':'3', 'i':'1', 'o':'0', 's':'$', 'l':'1'}

    word = ''.join(substitutions.get(c.lower(), c) for c in word)
    
    if random.random() < 0.2:
        word = '.'.join(word)
    
    if random.random() < 0.2:
        word = ''.join(c.upper() if random.random() > 0.5 else c.lower() for c in word)
    
    return word

In [15]:
neutral_words = [
    "meeting", "report", "team", "update", "business", "schedule",
    "note", "discussion", "project", "data", "information", "details",
    "client", "status", "review", "analysis", "system", "strategy", "plan"
]

In [16]:
emojis = ['🎯', '🔥', '✅', '📈', '💼', '🚀', '💰']

In [17]:
fake_signatures = [
    "\n\nBest regards,\nYour Team",
    "\n\nClick here to unsubscribe",
    "\n\nConfidential - For Internal Use Only",
    "\n\nSent from my iPhone",
    "\n\nAutomated Email - Do Not Reply",
]

In [18]:
def insert_neutral_words(text, rate=0.1):
    words = text.split()
    new_words = []
    for word in words:
        new_words.append(word)
        if random.random() < rate:
            neutral = random.choice(neutral_words)
            new_words.append(neutral)
    return ' '.join(new_words)

In [19]:
def insert_typos(text, typo_rate=0.05):
    chars = list(text)
    for i in range(len(chars)-1):
        if random.random() < typo_rate:
            chars[i], chars[i+1] = chars[i+1], chars[i]  # swap two letters
    return ''.join(chars)

In [20]:
def insert_emojis(text, emoji_rate=0.05):
    words = text.split()
    for i in range(len(words)):
        if random.random() < emoji_rate:
            emoji = random.choice(emojis)
            words[i] = emoji + ' ' + words[i]
    return ' '.join(words)

In [21]:

def add_fake_signature(text):
    if random.random() < 0.5:
        signature = random.choice(fake_signatures)
        return text + signature
    else:
        return text

In [22]:
# Adversarial Text Generator
def adversarial_transform(text, obfuscate_prob=0.3, dilution_rate=0.1, typo_rate=0.05, emoji_rate=0.05, aggressive=False):
    words = text.split()
    transformed = []
    
    for word in words:
        if random.random() < obfuscate_prob:
            transformed.append(obfuscate_word(word))
        else:
            transformed.append(word)
    
    adversarial_text = ' '.join(transformed)
    adversarial_text = insert_neutral_words(adversarial_text, rate=dilution_rate)
    
    if aggressive:
        adversarial_text = insert_typos(adversarial_text, typo_rate=typo_rate)
        adversarial_text = insert_emojis(adversarial_text, emoji_rate=emoji_rate)
        adversarial_text = add_fake_signature(adversarial_text)
        
    return adversarial_text

In [69]:
# Generate Multiple Versions
generated = []

for idx, row in df.iterrows():
    original = row['email']
    is_spam = row['target'] == "spam"

    if is_spam:
        # Generate adversarial versions for spam emails
        light = adversarial_transform(original, obfuscate_prob=0.1, dilution_rate=0.05, aggressive=False)
        medium = adversarial_transform(original, obfuscate_prob=0.2, dilution_rate=0.1, aggressive=False)
        heavy = adversarial_transform(original, obfuscate_prob=0.3, dilution_rate=0.15, typo_rate=0.1, emoji_rate=0.1, aggressive=True)
    else:
        # For ham: just duplicate original across all columns
        light = medium = heavy = original

    generated.append({
        'original': original,
        'adversarial_light': light,
        'adversarial_medium': medium,
        'adversarial_heavy': heavy,
        'target': row['target']
    })

In [70]:
# Generate Multiple Versions
generated2 = []

for idx, row in df2.iterrows():
    original = row['email']
    is_spam = row['target'] == "spam"

    if is_spam:
        # Generate adversarial versions for spam emails
        light = adversarial_transform(original, obfuscate_prob=0.1, dilution_rate=0.05, aggressive=False)
        medium = adversarial_transform(original, obfuscate_prob=0.2, dilution_rate=0.1, aggressive=False)
        heavy = adversarial_transform(original, obfuscate_prob=0.3, dilution_rate=0.15, typo_rate=0.1, emoji_rate=0.1, aggressive=True)
    else:
        # For ham: just duplicate original across all columns
        light = medium = heavy = original

    generated2.append({
        'original': original,
        'adversarial_light': light,
        'adversarial_medium': medium,
        'adversarial_heavy': heavy,
        'target': row['target']
    })

In [71]:
# Generate Multiple Versions
generated3 = []

for idx, row in df3.iterrows():
    original = row['email']
    is_spam = row['target'] == "spam"

    if is_spam:
        # Generate adversarial versions for spam emails
        light = adversarial_transform(original, obfuscate_prob=0.1, dilution_rate=0.05, aggressive=False)
        medium = adversarial_transform(original, obfuscate_prob=0.2, dilution_rate=0.1, aggressive=False)
        heavy = adversarial_transform(original, obfuscate_prob=0.3, dilution_rate=0.15, typo_rate=0.1, emoji_rate=0.1, aggressive=True)
    else:
        # For ham: just duplicate original across all columns
        light = medium = heavy = original

    generated3.append({
        'original': original,
        'adversarial_light': light,
        'adversarial_medium': medium,
        'adversarial_heavy': heavy,
        'target': row['target']
    })

In [72]:
# Create final DataFrame
adv_df = pd.DataFrame(generated)

In [73]:
adv2_df = pd.DataFrame(generated2)

In [74]:
adv3_df = pd.DataFrame(generated3)

In [75]:
adv_df

,original,adversarial_light,adversarial_medium,adversarial_heavy,target
0,"Oh right, ok. I'll make sure that i do loads o...","Oh right, ok. I'll make sure that i do loads o...","Oh right, ok. I'll make sure that i do loads o...","Oh right, ok. I'll make sure that i do loads o...",ham
1,I am in tirupur. call you da.,I am in tirupur. call you da.,I am in tirupur. call you da.,I am in tirupur. call you da.,ham
2,No that just means you have a fat head,No that just means you have a fat head,No that just means you have a fat head,No that just means you have a fat head,ham
3,"You have won ?1,000 cash or a ?2,000 prize! To...","You have won ?1,000 cash or a ?2,000 prize! To...","y0u have won ?1,000 cash or @ ?2,000 prize! T....","You reivew havew on ?1,000 C@$Ho r 💼 @ ?2,000 ...",spam
4,Come aftr &lt;DECIMAL&gt; ..now i m cleaning t...,Come aftr &lt;DECIMAL&gt; ..now i m cleaning t...,Come aftr &lt;DECIMAL&gt; ..now i m cleaning t...,Come aftr &lt;DECIMAL&gt; ..now i m cleaning t...,ham
...,...,...,...,...,...
1110,Stop the story. I've told him i've returned it...,Stop the story. I've told him i've returned it...,Stop the story. I've told him i've returned it...,Stop the story. I've told him i've returned it...,ham
1111,We have sent JD for Customer Service cum Accou...,We have sent JD for Customer Service cum Accou...,We have sent JD for Customer Service cum Accou...,We have sent JD for Customer Service cum Accou...,ham
1112,Just re read it and I have no shame but tell m...,Just re read it and I have no shame but tell m...,Just re read it and I have no shame but tell m...,Just re read it and I have no shame but tell m...,ham
1113,"Well, I meant as opposed to my drunken night o...","Well, I meant as opposed to my drunken night o...","Well, I meant as opposed to my drunken night o...","Well, I meant as opposed to my drunken night o...",ham


In [76]:
adv2_df

,original,adversarial_light,adversarial_medium,adversarial_heavy,target
0,have got * few things to do. may be in * pub l...,have got * few things to do. may be in * pub l...,have got * few things to do. may be in * pub l...,have got * few things to do. may be in * pub l...,ham
1,Okie but i scared u say i fat... Then u dun wa...,Okie but i scared u say i fat... Then u dun wa...,Okie but i scared u say i fat... Then u dun wa...,Okie but i scared u say i fat... Then u dun wa...,ham
2,Wot is u up 2 then bitch?,Wot is u up 2 then bitch?,Wot is u up 2 then bitch?,Wot is u up 2 then bitch?,ham
3,Ill be at yours in about 3 mins but look out f...,Ill be at yours in about 3 mins but look out f...,Ill be at yours in about 3 mins but look out f...,Ill be at yours in about 3 mins but look out f...,ham
4,Becoz its &lt;#&gt; jan whn al the post ofice ...,Becoz its &lt;#&gt; jan whn al the post ofice ...,Becoz its &lt;#&gt; jan whn al the post ofice ...,Becoz its &lt;#&gt; jan whn al the post ofice ...,ham
...,...,...,...,...,...
887,Cool. Do you like swimming? I have a pool and ...,Cool. Do you like swimming? I have a pool and ...,Cool. Do you like swimming? I have a pool and ...,Cool. Do you like swimming? I have a pool and ...,ham
888,1's reach home call me.,1's reach home call me.,1's reach home call me.,1's reach home call me.,ham
889,Not able to do anything.,Not able to do anything.,Not able to do anything.,Not able to do anything.,ham
890,Ok thanx... Take care then...,Ok thanx... Take care then...,Ok thanx... Take care then...,Ok thanx... Take care then...,ham


In [77]:
adv3_df

,original,adversarial_light,adversarial_medium,adversarial_heavy,target
0,What to think no one saying clearly. Ok leave ...,What to think no one saying clearly. Ok leave ...,What to think no one saying clearly. Ok leave ...,What to think no one saying clearly. Ok leave ...,ham
1,"FREE RING TONE just text \POLYS\"" to 87131. Th...","FREE RING TONE just text \POLYS\"" to 87131. Th...","FREE RING TONE just text \POLYS\"" to 87131. Th...","FREE RIGN ✅ lcient TONE ujst etxt \POLYS\"" to ...",spam
2,"Trust me. Even if isn't there, its there.","Trust me. Even if isn't there, its there.","Trust me. Even if isn't there, its there.","Trust me. Even if isn't there, its there.",ham
3,Hi dear we saw dear. We both are happy. Where ...,Hi dear we saw dear. We both are happy. Where ...,Hi dear we saw dear. We both are happy. Where ...,Hi dear we saw dear. We both are happy. Where ...,ham
4,"URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...","URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...","URGENT, 1MP0RT@NT INFORMATION F0R O2 USER. TOD...","URGNE,TI MPORTANT INFORAMTIO FNOR O2 U$3.R TOD...",spam
...,...,...,...,...,...
3560,Never y lei... I v lazy... Got wat? Dat day _ ...,Never y lei... I v lazy... Got wat? Dat day _ ...,Never y lei... I v lazy... Got wat? Dat day _ ...,Never y lei... I v lazy... Got wat? Dat day _ ...,ham
3561,"Just arrived, see you in a couple days &lt;3","Just arrived, see you in a couple days &lt;3","Just arrived, see you in a couple days &lt;3","Just arrived, see you in a couple days &lt;3",ham
3562,"Haha awesome, I might need to take you up on t...","Haha awesome, I might need to take you up on t...","Haha awesome, I might need to take you up on t...","Haha awesome, I might need to take you up on t...",ham
3563,I'm very happy for you babe ! Woo hoo party on...,I'm very happy for you babe ! Woo hoo party on...,I'm very happy for you babe ! Woo hoo party on...,I'm very happy for you babe ! Woo hoo party on...,ham


In [78]:
# Save it to target path
adv_df.to_csv('../dataset/sms/mixed/test_mixed.csv', index=False)

In [79]:
# Save it to target path
adv2_df.to_csv('../dataset/sms/mixed/val_mixed.csv', index=False)

In [80]:
# Save it to target path
adv3_df.to_csv('../dataset/sms/mixed/train_mixed.csv', index=False)